## <u>Bio Informer</u>
This project is an AI powered personal information assistant that uses a vector database to store and retrieve data from my resume. The resume is split into smaller chunks, embedded, and indexed for efficient semantic search. Through a tool based retrieval approach, the system fetches the most relevant information from the vector database in response to user queries and uses it to generate accurate, context aware answers. The application is designed as an agent that not only generates responses but also reviews the generated content and provides feedback on the quality and relevance of the information produced by the language model. By combining retrieval augmented generation, vector search, and tool calling, the application enables intelligent interaction with personal data, allowing users to easily access insights about my skills, experience, and background.

In [139]:
#Install Dependencies
import ensurepip
ensurepip.bootstrap()

Looking in links: /tmp/tmpskcqp9fi


In [140]:
#Upgrade pip
import sys
!{sys.executable} -m pip install --upgrade pip

In [141]:
#Install langchain-chroma
import sys
!{sys.executable} -m pip install langchain-chroma chromadb langchain langchain-openai langchain-huggingface

In [ ]:
#Imports
import os
import json
from dotenv import load_dotenv
from pypdf import PdfReader
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma  
from langchain_core.documents import Document
import requests
import gradio as gr

In [143]:
##Load Enviroment Variables
load_dotenv(override=True)
client = OpenAI()

In [144]:
#Configs
name = "Stephen Mwangi Muiyuro"
DB_NAME = "./vector_db"

In [145]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [146]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [147]:
#Extract Resume Text
reader = PdfReader("me/Resume.pdf")

resume_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume_text += text

In [148]:
#Chunk Resume Text
def chunk_text(text, chunk_size=500, overlap=100):
    step = chunk_size - overlap
    return [text[i:i + chunk_size] for i in range(0, len(text), step)]

chunks = chunk_text(resume_text)

In [149]:

# Initialize embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create documents
docs = [
    Document(
        page_content=chunk,
        metadata={"source": "resume", "doc_type": "cv"}
    )
    for chunk in chunks
]

# Create vector store
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=DB_NAME
)

In [150]:
#Retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 12,
        "lambda_mult": 0.5,
    },
)

In [151]:
#Format Context
def format_context(docs):
    parts = []

    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        doc_type = doc.metadata.get("doc_type", "unknown")

        parts.append(f"""
Document {i}
Source: {source}
Type: {doc_type}

{doc.page_content}
""")

    return "\n".join(parts)

In [152]:
def rag_retriever(query: str) -> str:
    docs = retriever.invoke(query)
    print("\n Retrieved Docs:")
 
    if not docs:
        return "I don't have that information."

    context = format_context(docs)

    prompt = f"""
You are {name}.

Answer the question clearly and professionally using ONLY the context below.
If the answer is not in the context, say you don't know.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [153]:
#Evaluation Model
class Evaluation(BaseModel):
    clarity: Literal["awesome", "good", "fair", "bad"]
    strength: Literal["strong", "moderate", "weak"]
    score: int

In [154]:
#Feedback Evaluator
def feedback_evaluator(response: str, query: str) -> Evaluation:
    judge_prompt = f"""
You are an evaluator reviewing an AI response.

User Query:
{query}

Model Response:
{response}

Evaluate based on:
 Clarity (awesome, good, fair, bad)
 Strength (strong, moderate, weak)
 Score (0-100)

Return in required schema.
"""

    eval_response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": judge_prompt}],
        response_format=Evaluation
    )

    return eval_response.choices[0].message.parsed

In [155]:
#Record User Details
def record_user_details(email: str, name: str = "Name not provided"):
    push(f"{name} of {email} has expressed interest")
    return "User details recorded."


def record_unknown_question(question: str):
    push(f"I'm unable to respond to this question: {question}")
    return "Question recorded."

In [156]:
#Tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "rag_retriever",
            "description": "Retrieve information about Stephen Mwangi Muiyuro from resume",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "record_user_details",
            "description": "Store user contact details",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string"},
                    "name": {"type": "string"}
                },
                "required": ["email"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "record_unknown_question",
            "description": "Store unknown questions",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"}
                },
                "required": ["question"]
            }
        }
    }
]

In [157]:
def handle_tool_calls(tool_calls):
    results = []

    TOOL_REGISTRY = {
        "rag_retriever": rag_retriever,
        "record_user_details": record_user_details,
        "record_unknown_question": record_unknown_question,
    }

    for tool_call in tool_calls:
        tool_name = tool_call.function.name

        try:
            arguments = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError:
            arguments = {}

        print(f"Tool called: {tool_name}", flush=True)

        tool = TOOL_REGISTRY.get(tool_name)

        try:
            if tool:
                result = tool(**arguments)
            else:
                result = f"Tool '{tool_name}' not found."
        except Exception as e:
            result = f"Error executing tool '{tool_name}': {str(e)}"

        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })

    return results

In [167]:
system_prompt = f"""
You are acting as {name}.

You are conversational, professional, and engaging.

IMPORTANT RULES:
- Only call rag_retriever when the question is clearly about your career, skills, background, or experience.
- Do NOT call rag_retriever for greetings.
- For greetings or small talk, respond naturally like a human.

- If you don't know something career-related to {name}:
    call record_unknown_question

- If the user shows interest:
    ask for email and call record_user_details

Stay in character as {name}.
"""

In [169]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [
        {"role": "user", "content": message}
    ]

    max_retries = 3
    attempts = []

    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        message_obj = response.choices[0].message

        if message_obj.tool_calls:
            messages.append(message_obj)
            tool_results = handle_tool_calls(message_obj.tool_calls)
            messages.extend(tool_results)
            continue

        current_answer = message_obj.content

        if "I don't have that information" in current_answer:
            messages.append({
                "role": "user",
                "content": f"Record this unknown question: {message}"
            })

            follow_up = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                tools=tools,
                tool_choice="auto"
            )

            follow_message = follow_up.choices[0].message

            if follow_message.tool_calls:
                messages.append(follow_message)
                tool_results = handle_tool_calls(follow_message.tool_calls)
                messages.extend(tool_results)

            return current_answer

        for i in range(max_retries):
            evaluation = feedback_evaluator(
                response=current_answer,
                query=message
            )

            print(f"Attempt {i + 1} Evaluation:", evaluation)

            attempts.append({
                "answer": current_answer,
                "score": evaluation.score
            })

            if evaluation.score >= 85:
                return current_answer

            messages.append({
                "role": "system",
                "content": f"""
Your previous answer was not good enough.

Evaluation:
Clarity: {evaluation.clarity}
Strength: {evaluation.strength}
Score: {evaluation.score}

Rewrite the answer to improve clarity, depth, and professionalism.
Make it more complete and better structured.
"""
            })

            retry_response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages
            )

            current_answer = retry_response.choices[0].message.content

       
        best_attempt = max(attempts, key=lambda x: x["score"])

        print("Best score selected:", best_attempt["score"])

        return best_attempt["answer"]

In [ ]:

ui = gr.ChatInterface(
    fn=chat,
    type="messages"
)

ui.launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


Attempt 1 Evaluation: clarity='awesome' strength='strong' score=95
Tool called: rag_retriever

 Retrieved Docs:
Attempt 1 Evaluation: clarity='awesome' strength='strong' score=95
Tool called: record_user_details
Push: Steve of stevemuiyuro@gmail.com has expressed interest
Attempt 1 Evaluation: clarity='awesome' strength='strong' score=90
Tool called: rag_retriever

 Retrieved Docs:
Attempt 1 Evaluation: clarity='awesome' strength='strong' score=90
